# Using Optuna for hyperparameter tuning

# Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, LabelEncoder, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
import optuna
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb

In [2]:
o = pd.read_csv('/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv')
df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

In [3]:
df = pd.concat([df,o])
df.drop(columns = 'id',inplace = True)
df.sample(5)

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
244793,Sandy,6.03,21.67,0.78,0.69,32.66,70.51,1784.74,9.03,1.95,Potato,Flowering,Zaid,Drip,River,3.95,No,100.06,East,Medium
178529,Silt,5.26,13.69,0.98,1.32,26.97,82.36,1316.45,9.09,19.73,Maize,Vegetative,Kharif,Rainfed,River,6.03,Yes,78.18,East,Medium
506865,Silt,8.13,51.38,0.35,0.35,18.00,30.47,1718.18,10.08,4.68,Cotton,Harvest,Kharif,Canal,River,11.24,No,35.69,West,Low
28948,Clay,5.55,35.85,0.42,0.14,41.61,82.21,2336.06,6.11,19.45,Rice,Flowering,Kharif,Canal,Rainwater,12.81,No,71.11,South,Medium
462037,Sandy,6.51,51.32,0.81,0.73,17.52,55.16,1735.72,6.34,16.54,Potato,Flowering,Rabi,Sprinkler,Rainwater,14.30,Yes,0.98,West,Low


# Feature Engineering

In [4]:
def add_cols(df):
    df['Moisture_Deficit'] = (40 - df['Soil_Moisture']).clip(lower=0)
    df['Weather_Stress'] = (
    df['Temperature_C'] * 0.3 +
    df['Wind_Speed_kmh'] * 0.2 -
    df['Humidity'] * 0.2 -
    df['Rainfall_mm'] * 0.3
    )

    stage_map = {
    "Sowing": 0,
    "Vegetative": 1,
    "Flowering": 2,
    "Harvest": 3
    }
    df['Stage_Weight'] = df['Crop_Growth_Stage'].map(stage_map)
    df['Rain_SoilMoisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
    df['Mulch_Effect'] = (df['Mulching_Used'] == 'Yes').astype(int) * (1 - df['Soil_Moisture']/100)
    stage_mult = {'Sowing': 0.8, 'Vegetative': 1.2, 'Flowering': 1.8, 'Harvest': 0.6}
    df['Stage_Water_Multiplier'] = df['Crop_Growth_Stage'].map(stage_mult)
    return df

In [5]:
df = add_cols(df)
test = add_cols(test)
# df = df.fillna(0)

In [6]:
pd.set_option('display.max_columns', None)
df.sample(5)

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,Moisture_Deficit,Weather_Stress,Stage_Weight,Rain_SoilMoisture,Mulch_Effect,Stage_Water_Multiplier
143723,Clay,6.12,39.08,0.90,0.81,37.58,34.79,1281.39,6.24,8.08,Rice,Harvest,Zaid,Canal,Groundwater,4.20,No,17.88,North,Low,0.92,-378.485,3,50076.7212,0.0000,0.6
547202,Silt,7.05,29.03,0.72,3.09,28.89,84.56,1729.94,6.83,10.23,Wheat,Sowing,Zaid,Rainfed,Reservoir,6.92,Yes,97.71,West,Low,10.97,-525.181,0,50220.1582,0.7097,0.8
620061,Loamy,7.80,43.86,0.51,1.71,22.36,50.62,2435.98,4.61,19.70,Maize,Harvest,Kharif,Drip,Groundwater,2.92,No,60.12,East,Low,0.00,-730.270,3,106842.0828,0.0000,0.6
312412,Clay,5.12,35.41,0.62,0.50,26.28,25.27,1800.63,5.71,18.46,Rice,Sowing,Rabi,Rainfed,Rainwater,3.59,No,20.65,East,Low,4.59,-533.667,0,63760.3083,0.0000,0.8
136891,Sandy,6.71,58.73,0.60,2.03,38.68,69.22,2138.30,8.64,2.48,Sugarcane,Sowing,Zaid,Drip,Rainwater,10.67,Yes,86.56,East,Low,0.00,-643.234,0,125582.3590,0.4127,0.8


# ColumnTransformer for OHE and Scaling

In [7]:
scaler_cols = df.select_dtypes(include = [int,float]).drop(columns = 'Stage_Weight').columns.tolist()
ohe_cols = df.select_dtypes(include = 'object').drop(columns = 'Irrigation_Need').columns.tolist()

In [8]:
clt = ColumnTransformer(transformers=[
    ('scaler',StandardScaler(),scaler_cols),
    ('ohe',OneHotEncoder(),ohe_cols)
],remainder='passthrough')
# clt = ColumnTransformer(transformers=[
#     ('scaler',MinMaxScaler(),scaler_cols),
#     ('ohe',OneHotEncoder(),ohe_cols)
# ],remainder='passthrough')

# Stacking

In [9]:
# data

X = df.drop(columns = 'Irrigation_Need')
X = clt.fit_transform(X)
y = df.Irrigation_Need
target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
y = y.map(target_mapping)
y = np.array(y)
test_ = test.drop(columns = 'id')
test_ = clt.transform(test_)
n_classes = len(np.unique(y))
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, {n_classes} classes")

Dataset: 640000 samples, 49 features, 3 classes


In [10]:
XGB = XGBClassifier(n_estimators = 831, max_depth = 4, learning_rate = 0.23590514823860093, min_child_weight = 3, gamma = 1.2456297224253232,
                         subsample = 0.7804018290363833, colsample_bytree = 0.7983030484833766, colsample_bylevel = 0.9241973849866497, reg_alpha = 0.2164667458085279,
                         reg_lambda = 6.479363279401208e-06,
                         objective = 'multi:softprob',
                         num_class = 3,
                         eval_metric = 'mlogloss',
                         random_state = 42,
                         device = 'cuda',
                         verbosity = 0,
                         class_weight = 'balanced')

In [11]:
# Level 1: 5 diverse high-performing base models
level1_estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_split=2,
        random_state=42, n_jobs=-1
    )),
    ('et', ExtraTreesClassifier(
        n_estimators=300, max_depth=None,
        random_state=43, n_jobs=-1
    )),
    ('cat', CatBoostClassifier(
        random_state=44,loss_function='MultiClass',
        eval_metric='MultiClass',
        auto_class_weights='Balanced'
    )),
    ('xgb1',XGB)
]

In [12]:
# Level 2: 3 strong meta-models (also diverse)
level2_estimators = [
    ('xgb2', XGB),
    ('lgb', lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(np.unique(y)),
        metric='multi_logloss',
        boosting_type='gbdt', 
        num_leaves=31,
        max_depth=-1,
        learning_rate=0.05,
        n_estimators=500,
        random_state=42,
        verbosity=-1,
        class_weight='balanced'
    ))
]

In [13]:
# Final layer: 1 very strong meta-learner (XGBoost)
final_model = XGB

In [14]:
level2_stack = StackingClassifier(
    estimators=level2_estimators,
    final_estimator=final_model,
    cv=2,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False
)

In [15]:
final_ensemble = StackingClassifier(
    estimators=level1_estimators,
    final_estimator=level2_stack,
    cv=2,
    stack_method='predict_proba',
    n_jobs=-1,
    passthrough=False
)

In [16]:
final_ensemble.fit(X, y)

Learning rate set to 0.110012
0:	learn: 0.9114079	total: 1.18s	remaining: 19m 42s
1:	learn: 0.7747039	total: 2.23s	remaining: 18m 30s
2:	learn: 0.6698846	total: 3.52s	remaining: 19m 28s
3:	learn: 0.5881760	total: 4.48s	remaining: 18m 35s
4:	learn: 0.5197417	total: 5.43s	remaining: 18m 1s
5:	learn: 0.4637878	total: 6.55s	remaining: 18m 4s
6:	learn: 0.4166487	total: 7.71s	remaining: 18m 14s
7:	learn: 0.3763496	total: 8.7s	remaining: 17m 58s
8:	learn: 0.3423056	total: 9.72s	remaining: 17m 49s
9:	learn: 0.3127081	total: 10.6s	remaining: 17m 34s
10:	learn: 0.2875868	total: 11.7s	remaining: 17m 33s
11:	learn: 0.2658622	total: 12.8s	remaining: 17m 32s
12:	learn: 0.2467673	total: 13.7s	remaining: 17m 18s
13:	learn: 0.2299519	total: 14.5s	remaining: 17m 1s
14:	learn: 0.2148521	total: 15.5s	remaining: 16m 55s
15:	learn: 0.2024437	total: 16.4s	remaining: 16m 47s
16:	learn: 0.1911418	total: 17.3s	remaining: 16m 40s
17:	learn: 0.1809611	total: 18.3s	remaining: 16m 37s
18:	learn: 0.1725572	total: 19

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


StackingClassifier(cv=2,
                   estimators=[('rf',
                                RandomForestClassifier(n_estimators=300,
                                                       n_jobs=-1,
                                                       random_state=42)),
                               ('et',
                                ExtraTreesClassifier(n_estimators=300,
                                                     n_jobs=-1,
                                                     random_state=43)),
                               ('cat',
                                CatBoostClassifier(auto_class_weights='Balanced', eval_metric='MultiClass', loss_function='MultiClass', random_state=44)),
                               ('xgb1',
                                XGBClassifier(base_score=None, boos...
                                                                                    interaction_constraints=None,
                                                                                    learning_rate=0.23590514823860093,
                                                                                    max_bin=None,
                                                                                    max_cat_threshold=None,
                                                                                    max_cat_to_onehot=None,
                                                                                    max_delta_step=None,
                                                                                    max_depth=4,
                                                                                    max_leaves=None,
                                                                                    min_child_weight=3,
                                                                                    missing=nan,
                                                                                    monotone_constraints=None,
                                                                                    multi_strategy=None,
                                                                                    n_estimators=831,
                                                                                    n_jobs=None, ...),
                                                      n_jobs=-1,
                                                      stack_method='predict_proba'),
                   n_jobs=-1, stack_method='predict_proba')

In [17]:
y_pred = final_ensemble.predict(test_)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [18]:
y_pred = np.where(y_pred == 0, 'Low',
         np.where(y_pred == 1, 'Medium', 'High'))
submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': y_pred
})
submission.to_csv('1submission.csv', index=False)